In [3]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/Ahmed0000712/Banking-Fraud-Detection-and-Investigation/main"

customers = pd.read_csv(f"{BASE}/Data/raw/customers.csv")
accounts  = pd.read_csv(f"{BASE}/Data/raw/accounts.csv")
tx        = pd.read_csv(f"{BASE}/Data/raw/transactions.csv", parse_dates=["Timestamp"])
logins    = pd.read_csv(f"{BASE}/Data/raw/login_activity.csv", parse_dates=["Timestamp"])
answer    = pd.read_csv(f"{BASE}/Data/answer_key/answer_key.csv")

In [5]:
tables = {"customers": customers, "accounts": accounts,
          "transactions": tx, "logins": logins, "answer_key": answer}

In [6]:
for name, df in tables.items():
    print(f"{name:13} missing: {df.isna().sum().sum()}  duplicates: {df.duplicated().sum()}")

customers     missing: 0  duplicates: 0
accounts      missing: 0  duplicates: 0
transactions  missing: 0  duplicates: 0
logins        missing: 0  duplicates: 0
answer_key    missing: 0  duplicates: 0


In [7]:
def check(name, ok):
    print("PASS" if ok else "FAIL", "-", name)

owner = accounts.set_index("AccountID")["CustomerID"]

check("CustomerID unique",        customers["CustomerID"].is_unique)
check("AccountID unique",         accounts["AccountID"].is_unique)
check("TransactionID unique",     tx["TransactionID"].is_unique)
check("LoginID unique",           logins["LoginID"].is_unique)
check("Accounts have customers",  accounts["CustomerID"].isin(customers["CustomerID"]).all())
check("Senders exist",            tx["SenderAccount"].isin(accounts["AccountID"]).all())
check("Receivers exist",          tx["ReceiverAccount"].isin(accounts["AccountID"]).all())
check("Login accounts exist",     logins["AccountID"].isin(accounts["AccountID"]).all())
check("Tx owner matches sender",  (tx["SenderAccount"].map(owner) == tx["CustomerID"]).all())

PASS - CustomerID unique
PASS - AccountID unique
PASS - TransactionID unique
PASS - LoginID unique
PASS - Accounts have customers
PASS - Senders exist
PASS - Receivers exist
PASS - Login accounts exist
PASS - Tx owner matches sender


In [8]:
check("Sender != receiver",     (tx["SenderAccount"] != tx["ReceiverAccount"]).all())
check("Amount > 0",             (tx["Amount"] > 0).all())
check("Age 18 to 100",          customers["Age"].between(18, 100).all())
check("Failed attempts >= 0",   (logins["FailedAttempts"] >= 0).all())
check("Balance >= 0",           (accounts["OpeningBalance"] >= 0).all())
check("Valid tx type",          tx["TransactionType"].isin(["UPI","IMPS","NEFT","RTGS"]).all())
check("Answer key tx exist",    answer.loc[answer.EntityType=="TRANSACTION","RecordID"].isin(tx["TransactionID"]).all())
check("Answer key login exist", answer.loc[answer.EntityType=="LOGIN","RecordID"].isin(logins["LoginID"]).all())

PASS - Sender != receiver
PASS - Amount > 0
PASS - Age 18 to 100
PASS - Failed attempts >= 0
PASS - Balance >= 0
PASS - Valid tx type
PASS - Answer key tx exist
PASS - Answer key login exist


In [9]:
print("Transactions:", len(tx), "| Logins:", len(logins))
print("Dates:", tx["Timestamp"].min(), "to", tx["Timestamp"].max())
print("Fraud transactions:", (answer.EntityType == "TRANSACTION").sum())
print("Logins per customer (min/max):",
      logins.groupby("CustomerID").size().min(), "/",
      logins.groupby("CustomerID").size().max())

Transactions: 814 | Logins: 102
Dates: 2026-08-01 07:19:00 to 2026-10-29 21:20:00
Fraud transactions: 14
Logins per customer (min/max): 1 / 2
